In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from attention import basic_attention, einsum_attention, stream_attention

In [3]:
import torch

torch.manual_seed(123)
Q= torch.randn(2, 4, 8)  # (batch_size, seq_len_q, d_k)
K= torch.randn(2, 5, 8)  # (batch_size, seq_len_k, d_k)
V= torch.randn(2, 5, 8)  # (batch_size, seq_len_v, d_v)

output_basic = basic_attention(Q, K, V)
output_einops = einsum_attention(Q, K, V)

In [4]:
output_basic.shape, output_einops.shape

(torch.Size([2, 4, 8]), torch.Size([2, 4, 8]))

In [5]:
torch.allclose(output_basic, output_einops)

True

In [6]:
# %timeit basic_attention(Q, K, V)

In [7]:
# %timeit einsum_attention(Q, K, V)

In [8]:
import torch
import torch.utils.benchmark as benchmark

device = "cuda" if torch.cuda.is_available() else "cpu"
results = []

# Use realistic dimensions: batch, heads, sequence, head dimension
shapes = [
    (2, 4, 128, 64),
    (2, 4, 256, 64),
    (2, 4, 512, 64),
]

for shape in shapes:
    Q = torch.randn(shape, device=device)
    K = torch.randn(shape, device=device)
    V = torch.randn(shape, device=device)

    # Verify numerical equivalence first
    torch.testing.assert_close(
        basic_attention(Q, K, V),
        einsum_attention(Q, K, V),
    )

    for name, fn in [
        ("matmul", basic_attention),
        ("einsum", einsum_attention),
    ]:
        timer = benchmark.Timer(
            stmt="fn(Q, K, V)",
            globals={"fn": fn, "Q": Q, "K": K, "V": V},
            label="Attention forward",
            sub_label=str(shape),
            description=name,
        )
        results.append(timer.blocked_autorange(min_run_time=1))

benchmark.Compare(results).print()

[----------- Attention forward -----------]
                       |  matmul  |  einsum
1 threads: --------------------------------
      (2, 4, 128, 64)  |  103.9   |  160.8 
      (2, 4, 256, 64)  |  100.1   |  166.9 
      (2, 4, 512, 64)  |  365.5   |  368.3 

Times are in microseconds (us).



In [9]:
# Test non-batched streaming attention, including d_v != d_k
torch.manual_seed(123)
Q = torch.randn(4, 8, dtype=torch.float64)
K = torch.randn(6, 8, dtype=torch.float64)
V = torch.randn(6, 5, dtype=torch.float64)

expected = basic_attention(Q, K, V)
actual = stream_attention(Q, K, V, q_block_size=3, kv_block_size=2)
torch.testing.assert_close(actual, expected, rtol=1e-10, atol=1e-12)

print("stream_attention forward test passed")

stream_attention forward test passed
